# Tune Random Forest Winner Prediction

Tune Random Forest parameters using five stratified random 90/10 splits. The feature set is fixed to the best current baseline: `DIFF + LOGDIFF volume + ABS Elo`.

In [1]:
from pathlib import Path
from time import perf_counter

import numpy as np
import pandas as pd

from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.metrics import accuracy_score, roc_auc_score
from sklearn.model_selection import ParameterSampler, StratifiedShuffleSplit
from sklearn.pipeline import Pipeline


RANDOM_STATE = 42
N_RANDOM_SPLITS = 5
TEST_SIZE = 0.10
N_PARAM_SAMPLES = 60

data_path = Path("../data_clean/pre_fight_career_stats.csv")
if not data_path.exists():
    data_path = Path("data_clean/pre_fight_career_stats.csv")

df = pd.read_csv(data_path)
df["date"] = pd.to_datetime(df["date"], errors="coerce")
df = df[df["winner_side"].isin(["R", "B"])].copy()
df = df.dropna(subset=["date", "winner_side"]).reset_index(drop=True)

print("Loaded:", data_path)
print("Shape:", df.shape)
print(df["winner_side"].value_counts())
df.head()

Loaded: ..\data_clean\pre_fight_career_stats.csv
Shape: (3032, 119)
winner_side
R    1823
B    1209
Name: count, dtype: int64


,fight_id,date,R_fighter,B_fighter,winner,winner_side,win_by,R_elo,B_elo,R_total_fights,...,B_DISTANCE_absorbed_per_min,B_DISTANCE_opp_att_per_min,B_CLINCH_land_per_min,B_CLINCH_att_per_min,B_CLINCH_absorbed_per_min,B_CLINCH_opp_att_per_min,B_GROUND_land_per_min,B_GROUND_att_per_min,B_GROUND_absorbed_per_min,B_GROUND_opp_att_per_min
0,8632,1995-12-16,Dan Severn,Oleg Taktarov,Dan Severn,R,Decision - Unanimous,1572.732116,1579.940002,8,...,0.581583,2.132472,0.038772,0.135703,0.213247,0.465267,0.000000,0.000000,0.678514,1.124394
1,8633,1995-12-16,Oleg Taktarov,Dave Beneteau,Oleg Taktarov,R,Submission,1559.525142,1503.357099,8,...,0.000000,0.000000,0.920716,2.455243,0.153453,0.613811,3.222506,5.217391,0.153453,0.153453
2,8635,1995-12-16,Dan Severn,Paul Varelans,Dan Severn,R,Submission,1593.146976,1518.980366,9,...,1.912568,3.606557,0.491803,0.765027,0.327869,1.366120,0.327869,0.327869,0.928962,1.475410
3,8616,1996-05-17,Dan Severn,Ken Shamrock,Dan Severn,R,Decision - Split,1608.941292,1581.067177,10,...,0.205781,0.911318,0.235179,0.293974,0.146987,0.235179,1.028907,1.499265,0.146987,0.205781
4,8595,1996-12-07,Ken Shamrock,Brian Johnston,Ken Shamrock,R,Submission,1562.668306,1507.774652,7,...,0.203390,0.508475,1.220339,1.830508,0.508475,0.711864,0.711864,1.118644,1.627119,1.830508


## Balance Side Labels

Randomly flip fight orientation so `winner_side` is as close to 50/50 as possible within each year.

In [2]:
def find_red_blue_pairs(columns):
    pairs = []
    for col in columns:
        if col.startswith("R_"):
            b_col = "B_" + col[2:]
            if b_col in columns:
                pairs.append((col, b_col))
    return pairs


def swap_red_blue_rows(data, swap_mask):
    balanced = data.copy()
    pairs = find_red_blue_pairs(balanced.columns)
    r_cols = [r for r, _ in pairs]
    b_cols = [b for _, b in pairs]

    r_values = balanced.loc[swap_mask, r_cols].to_numpy(copy=True)
    b_values = balanced.loc[swap_mask, b_cols].to_numpy(copy=True)
    balanced.loc[swap_mask, r_cols] = b_values
    balanced.loc[swap_mask, b_cols] = r_values

    balanced.loc[swap_mask, "winner_side"] = balanced.loc[swap_mask, "winner_side"].map({"R": "B", "B": "R"})
    return balanced


def randomly_balance_winner_side(data, random_state=42, balance_by_year=True):
    rng = np.random.default_rng(random_state)
    balanced = data.copy().reset_index(drop=True)

    desired_side = pd.Series(index=balanced.index, dtype="object")
    groups = balanced["date"].dt.year if balance_by_year else pd.Series(0, index=balanced.index)

    for _, group_idx in balanced.groupby(groups).groups.items():
        group_idx = np.array(list(group_idx))
        n_rows = len(group_idx)
        group_sides = ["R"] * (n_rows // 2) + ["B"] * (n_rows // 2)

        if n_rows % 2 == 1:
            group_sides.append(rng.choice(["R", "B"]))

        group_sides = np.array(group_sides)
        rng.shuffle(group_sides)
        desired_side.loc[group_idx] = group_sides

    swap_mask = balanced["winner_side"].to_numpy() != desired_side.to_numpy()
    balanced = swap_red_blue_rows(balanced, swap_mask)
    balanced["target"] = (balanced["winner_side"] == "R").astype(int)
    return balanced


balanced_df = randomly_balance_winner_side(df, random_state=RANDOM_STATE)

print("Original target distribution:")
print(df["winner_side"].value_counts())
print("\nBalanced target distribution:")
print(balanced_df["winner_side"].value_counts())

balanced_df.sort_values("date").groupby(balanced_df["date"].dt.year)["winner_side"].value_counts(normalize=True).unstack().tail(10)

Original target distribution:
winner_side
R    1823
B    1209
Name: count, dtype: int64

Balanced target distribution:
winner_side
R    1519
B    1513
Name: count, dtype: int64


winner_side,B,R
date,,
2017,0.497110,0.502890
2018,0.500000,0.500000
2019,0.497207,0.502793
2020,0.502924,0.497076
2021,0.497608,0.502392
2022,0.500000,0.500000
2023,0.500000,0.500000
2024,0.500000,0.500000
2025,0.502075,0.497925


## Build Best Feature Set

Create `DIFF_...`, `LOGDIFF_...` for volume/count/rate stats, and `ABS_DIFF_elo`.

In [3]:
def is_logdiff_candidate(suffix):
    skip_terms = ["pct", "acc", "def", "quickness", "elo"]
    return not any(term in suffix.lower() for term in skip_terms)


def add_matchup_features(data):
    new_features = {}

    for r_col, b_col in find_red_blue_pairs(data.columns):
        suffix = r_col[2:]
        if not (pd.api.types.is_numeric_dtype(data[r_col]) and pd.api.types.is_numeric_dtype(data[b_col])):
            continue

        diff_col = f"DIFF_{suffix}"
        new_features[diff_col] = data[r_col] - data[b_col]

        if is_logdiff_candidate(suffix):
            r_values = data[r_col].clip(lower=0)
            b_values = data[b_col].clip(lower=0)
            new_features[f"LOGDIFF_{suffix}"] = np.log1p(r_values) - np.log1p(b_values)

        if suffix == "elo":
            new_features["ABS_DIFF_elo"] = new_features[diff_col].abs()

    return pd.concat([data.copy(), pd.DataFrame(new_features, index=data.index)], axis=1)


model_df = add_matchup_features(balanced_df)
y = model_df["target"]

diff_cols = [col for col in model_df.columns if col.startswith("DIFF_")]
logdiff_cols = [col for col in model_df.columns if col.startswith("LOGDIFF_")]
abs_elo_cols = [col for col in ["ABS_DIFF_elo"] if col in model_df.columns]

feature_cols = list(dict.fromkeys(diff_cols + logdiff_cols + abs_elo_cols))
X = model_df[feature_cols].replace([np.inf, -np.inf], np.nan)

print("Rows:", len(model_df))
print("Features:", len(feature_cols))
feature_cols[:15]

Rows: 3032
Features: 98


['DIFF_elo',
 'DIFF_total_fights',
 'DIFF_total_fight_time',
 'DIFF_fight_quickness',
 'DIFF_win_DEC_pct',
 'DIFF_win_KO_TKO_pct',
 'DIFF_win_SUB_pct',
 'DIFF_loss_DEC_pct',
 'DIFF_loss_KO_TKO_pct',
 'DIFF_loss_SUB_pct',
 'DIFF_TOTAL_STR_acc',
 'DIFF_SIG_STR_acc',
 'DIFF_TD_acc',
 'DIFF_CTRL_pct',
 'DIFF_KD_per_min']

## Fixed 90/10 Splits

Every parameter set is evaluated on the same five splits.

In [4]:
splitter = StratifiedShuffleSplit(
    n_splits=N_RANDOM_SPLITS,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
)

splits = list(splitter.split(X, y))

for split_num, (train_idx, test_idx) in enumerate(splits, start=1):
    train_dist = y.iloc[train_idx].value_counts(normalize=True).rename({1: "R", 0: "B"})
    test_dist = y.iloc[test_idx].value_counts(normalize=True).rename({1: "R", 0: "B"})
    print(f"Split {split_num}: train={len(train_idx)} test={len(test_idx)}")
    print("  train target:", train_dist.to_dict())
    print("  test target:", test_dist.to_dict())

Split 1: train=2728 test=304
  train target: {'R': 0.501099706744868, 'B': 0.49890029325513197}
  test target: {'R': 0.5, 'B': 0.5}
Split 2: train=2728 test=304
  train target: {'R': 0.501099706744868, 'B': 0.49890029325513197}
  test target: {'R': 0.5, 'B': 0.5}
Split 3: train=2728 test=304
  train target: {'R': 0.501099706744868, 'B': 0.49890029325513197}
  test target: {'R': 0.5, 'B': 0.5}
Split 4: train=2728 test=304
  train target: {'R': 0.501099706744868, 'B': 0.49890029325513197}
  test target: {'R': 0.5, 'B': 0.5}
Split 5: train=2728 test=304
  train target: {'R': 0.501099706744868, 'B': 0.49890029325513197}
  test target: {'R': 0.5, 'B': 0.5}


## Random Forest Tuning

Use randomized parameter sampling to keep runtime reasonable. Increase `N_PARAM_SAMPLES` for a deeper search.

In [5]:
def make_rf_pipeline(params, random_state):
    return Pipeline(
        steps=[
            ("imputer", SimpleImputer(strategy="median")),
            (
                "model",
                RandomForestClassifier(
                    **params,
                    n_jobs=-1,
                    random_state=random_state,
                ),
            ),
        ]
    )


def evaluate_params(params, splits):
    accuracies = []
    aucs = []

    for split_num, (train_idx, test_idx) in enumerate(splits, start=1):
        model = make_rf_pipeline(params, random_state=RANDOM_STATE + split_num)
        model.fit(X.iloc[train_idx], y.iloc[train_idx])

        pred = model.predict(X.iloc[test_idx])
        proba = model.predict_proba(X.iloc[test_idx])[:, 1]

        accuracies.append(accuracy_score(y.iloc[test_idx], pred))
        aucs.append(roc_auc_score(y.iloc[test_idx], proba))

    return accuracies, aucs


baseline_params = {
    "n_estimators": 600,
    "min_samples_leaf": 8,
    "min_samples_split": 2,
    "max_depth": None,
    "max_features": "sqrt",
    "bootstrap": True,
    "criterion": "gini",
    "class_weight": None,
}

param_distributions = {
    "n_estimators": [300, 500, 700, 900, 1200],
    "max_depth": [None, 6, 10, 14, 20, 30],
    "max_features": ["sqrt", "log2", 0.25, 0.4, 0.6],
    "min_samples_leaf": [1, 2, 4, 6, 8, 12, 16],
    "min_samples_split": [2, 5, 10, 15, 20, 30],
    "bootstrap": [True, False],
    "criterion": ["gini", "entropy", "log_loss"],
    "class_weight": [None, "balanced_subsample"],
}

sampled_params = list(ParameterSampler(
    param_distributions,
    n_iter=N_PARAM_SAMPLES,
    random_state=RANDOM_STATE,
))

param_sets = [baseline_params] + sampled_params
print("Parameter sets to evaluate:", len(param_sets))

Parameter sets to evaluate: 61


In [6]:
results = []
start = perf_counter()

for i, params in enumerate(param_sets, start=1):
    accuracies, aucs = evaluate_params(params, splits)

    row = {
        "rank_input_order": i,
        "mean_accuracy": np.mean(accuracies),
        "std_accuracy": np.std(accuracies),
        "mean_auc": np.mean(aucs),
        "std_auc": np.std(aucs),
        "split_accuracies": accuracies,
        "split_aucs": aucs,
        **params,
    }
    results.append(row)

    if i == 1 or i % 10 == 0 or i == len(param_sets):
        elapsed = perf_counter() - start
        print(f"Evaluated {i}/{len(param_sets)} parameter sets in {elapsed:.1f}s")

tuning_results = pd.DataFrame(results).sort_values(
    ["mean_accuracy", "mean_auc"],
    ascending=False,
).reset_index(drop=True)

tuning_results.head(15)

Evaluated 1/61 parameter sets in 6.5s
Evaluated 10/61 parameter sets in 117.0s
Evaluated 20/61 parameter sets in 349.3s
Evaluated 30/61 parameter sets in 546.4s
Evaluated 40/61 parameter sets in 888.2s
Evaluated 50/61 parameter sets in 1136.7s
Evaluated 60/61 parameter sets in 1466.7s
Evaluated 61/61 parameter sets in 1483.0s


,rank_input_order,mean_accuracy,std_accuracy,mean_auc,std_auc,split_accuracies,split_aucs,n_estimators,min_samples_leaf,min_samples_split,max_depth,max_features,bootstrap,criterion,class_weight
0,1,0.606579,0.027174,0.635336,0.029463,"[0.6414473684210527, 0.5953947368421053, 0.565...","[0.641101108033241, 0.6112794321329641, 0.5931...",600,8,2,NaN,sqrt,True,gini,NaN
1,53,0.604605,0.022252,0.622126,0.021613,"[0.5986842105263158, 0.6118421052631579, 0.569...","[0.6197195290858726, 0.6185076177285318, 0.585...",900,16,5,6.0,0.25,False,entropy,NaN
2,19,0.603947,0.034096,0.627943,0.027236,"[0.6546052631578947, 0.5789473684210527, 0.559...","[0.634911703601108, 0.6068213296398892, 0.5876...",700,12,5,14.0,sqrt,False,log_loss,NaN
3,2,0.602632,0.014622,0.638591,0.025649,"[0.6085526315789473, 0.5921052631578947, 0.582...","[0.6351281163434903, 0.6093317174515235, 0.612...",300,2,15,14.0,sqrt,True,log_loss,NaN
4,6,0.602632,0.017579,0.634011,0.027690,"[0.6085526315789473, 0.5855263157894737, 0.578...","[0.6288521468144044, 0.608422783933518, 0.6011...",1200,1,20,30.0,0.25,True,log_loss,balanced_subsample
5,20,0.602632,0.034211,0.631899,0.026978,"[0.6414473684210527, 0.5888157894736842, 0.549...","[0.643395083102493, 0.6067780470914128, 0.5950...",900,8,10,14.0,log2,True,entropy,balanced_subsample
6,24,0.601974,0.026885,0.632245,0.027341,"[0.6151315789473685, 0.5789473684210527, 0.565...","[0.6350415512465374, 0.601843836565097, 0.6002...",700,1,15,10.0,0.4,True,gini,NaN
7,39,0.600658,0.006101,0.620282,0.012255,"[0.5953947368421053, 0.5921052631578947, 0.608...","[0.6111928670360111, 0.6175554016620498, 0.607...",500,12,10,20.0,0.6,False,gini,NaN
8,61,0.600000,0.018327,0.633717,0.023038,"[0.5921052631578947, 0.5855263157894737, 0.578...","[0.6277268005540166, 0.6172091412742383, 0.603...",500,12,30,NaN,0.6,True,gini,NaN
9,26,0.600000,0.015231,0.625814,0.018146,"[0.618421052631579, 0.6118421052631579, 0.5789...","[0.6278133656509695, 0.6181180747922439, 0.595...",300,16,30,10.0,0.25,False,entropy,balanced_subsample


## Best Model Summary

In [7]:
best = tuning_results.iloc[0]
baseline = tuning_results[tuning_results["rank_input_order"] == 1].iloc[0]

print("Best mean accuracy:", best["mean_accuracy"])
print("Best accuracy std:", best["std_accuracy"])
print("Best mean AUC:", best["mean_auc"])
print("\nBaseline mean accuracy:", baseline["mean_accuracy"])
print("Baseline mean AUC:", baseline["mean_auc"])
print("\nAccuracy lift:", best["mean_accuracy"] - baseline["mean_accuracy"])
print("AUC lift:", best["mean_auc"] - baseline["mean_auc"])

best_params = {
    "n_estimators": best["n_estimators"],
    "max_depth": best["max_depth"],
    "max_features": best["max_features"],
    "min_samples_leaf": best["min_samples_leaf"],
    "min_samples_split": best["min_samples_split"],
    "bootstrap": best["bootstrap"],
    "criterion": best["criterion"],
    "class_weight": best["class_weight"],
}

best_params

Best mean accuracy: 0.6065789473684211
Best accuracy std: 0.02717352161587287
Best mean AUC: 0.6353358725761773

Baseline mean accuracy: 0.6065789473684211
Baseline mean AUC: 0.6353358725761773

Accuracy lift: 0.0
AUC lift: 0.0


{'n_estimators': np.int64(600),
 'max_depth': np.float64(nan),
 'max_features': 'sqrt',
 'min_samples_leaf': np.int64(8),
 'min_samples_split': np.int64(2),
 'bootstrap': np.True_,
 'criterion': 'gini',
 'class_weight': nan}

In [8]:
output_path = Path("rf_tuning_results.csv")
tuning_results.to_csv(output_path, index=False)
print("Saved:", output_path)

tuning_results[[
    "mean_accuracy",
    "std_accuracy",
    "mean_auc",
    "std_auc",
    "n_estimators",
    "max_depth",
    "max_features",
    "min_samples_leaf",
    "min_samples_split",
    "bootstrap",
    "criterion",
    "class_weight",
]].head(20)

Saved: rf_tuning_results.csv


,mean_accuracy,std_accuracy,mean_auc,std_auc,n_estimators,max_depth,max_features,min_samples_leaf,min_samples_split,bootstrap,criterion,class_weight
0,0.606579,0.027174,0.635336,0.029463,600,NaN,sqrt,8,2,True,gini,NaN
1,0.604605,0.022252,0.622126,0.021613,900,6.0,0.25,16,5,False,entropy,NaN
2,0.603947,0.034096,0.627943,0.027236,700,14.0,sqrt,12,5,False,log_loss,NaN
3,0.602632,0.014622,0.638591,0.025649,300,14.0,sqrt,2,15,True,log_loss,NaN
4,0.602632,0.017579,0.634011,0.027690,1200,30.0,0.25,1,20,True,log_loss,balanced_subsample
5,0.602632,0.034211,0.631899,0.026978,900,14.0,log2,8,10,True,entropy,balanced_subsample
6,0.601974,0.026885,0.632245,0.027341,700,10.0,0.4,1,15,True,gini,NaN
7,0.600658,0.006101,0.620282,0.012255,500,20.0,0.6,12,10,False,gini,NaN
8,0.600000,0.018327,0.633717,0.023038,500,NaN,0.6,12,30,True,gini,NaN
9,0.600000,0.015231,0.625814,0.018146,300,10.0,0.25,16,30,False,entropy,balanced_subsample
